# 0. Introduction / Set-up

This notebook handles the sentiment analysis step for the fashion brand Everlane, which aims to answer the question: "Has Everlane's sustainability messaging become more concrete over time, or has it diverged from public perceptions of ethical practice?"

3 distinct sources were scraped to construct a multi-perspective corpus:
1. reddit_comments.csv and reddit_posts.csv - 4,600 Reddit posts and 12,400 comments covering consumer sentiment,
2. gdelt_articles_usable.csv - 28,400 lines from over 1,000 news articles capturing media coverage,
3. everlane_web.csv - nearly 1,400 lines scraped directly from Everlane's website. 

# 1. Configurations

I'm using `cardiffnlp/twitter-roberta-base-sentiment-latest` (RoBERTa trained on tweets, three classes: negative / neutral / positive) because most news and website sentences carry no opinion; binary models like `distilbert-base-uncased-finetuned-sst-2-english` were considered but have no neutral class, so they would force every factual sentence into positive or negative.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

# Paths
# The evaluation set is sampled and frozen in ner.ipynb (section 3); this notebook only adds sentiment labels to it
PROJECT_ROOT = Path.cwd().parent
EVAL_PATH    = PROJECT_ROOT / "data" / "eval" / "eval_set.csv"

# Model (id, revision; pin to commit hash once final)
SENTIMENT_MODEL  = ("cardiffnlp/twitter-roberta-base-sentiment-latest", "main")
SENTIMENT_LABELS = ["negative", "neutral", "positive"]   # the model's own label names

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# 2. Gold Sentiment Labels

Before running the model, I hand-label the same 152 frozen sentences used for NER, so the model is scored against labels I wrote without seeing its output. Each sentence gets one label, judged from the sentence alone:

- **positive**: praise, approval, satisfaction, excitement ("We ended up choosing them because of their top-notch production methods")
- **negative**: criticism, complaint, disappointment, anger ("They were terrible and falsely hyped.")
- **neutral**: facts, questions, instructions, fragments, no evaluative words ("How does it work?", "Espadrilles")

Edge cases:
- A factual claim that sounds good is still **neutral** unless it uses evaluative words ("they use biomass pellets" = neutral; "top-notch" = positive).
- Mixed sentences get the stronger side; if truly balanced, **neutral** and mark unsure.
- Sarcasm gets its intended meaning ("Great, another 'sustainable' drop" = negative).
- Sentiment toward anything counts, not only toward Everlane, because that is what the model measures.
- Quotes and reported speech in news are labeled by the tone of the sentence ("was painful" = negative).

The labeling cell hides the source and split so they can't bias the label, saves after every answer, and resumes where it left off. It has to run locally (on Colab it would save into the temporary clone).

In [ ]:
assert EVAL_PATH.exists(), "Run section 3 of ner.ipynb first to create the frozen eval set"
eval_df = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)

if "gold_sentiment" not in eval_df:   # adds a column only; never re-samples
    eval_df["gold_sentiment"] = ""
    eval_df.to_csv(EVAL_PATH, index=False)

print(f"{len(eval_df)} sentences, {eval_df['gold_sentiment'].ne('').sum()} labeled")

In [ ]:
# Label one sentence at a time: n = negative, u = neutral, p = positive
# Add "?" to flag an unsure call (e.g. "u?"), s = skip, q = quit and resume later
KEYS = {"n": "negative", "u": "neutral", "p": "positive"}

def save_label(uid, col, value, append=False):
    # Re-read the file before every write: ner.ipynb and sentiment.ipynb both label this file,
    # so writing back an in-memory copy would erase the other notebook's labels
    d = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)
    row = d["uid"] == uid
    if append:
        value = "; ".join(filter(None, [d.loc[row, col].iat[0], value]))
    d.loc[row, col] = value
    d.to_csv(EVAL_PATH, index=False)

eval_df = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)
todo = eval_df[eval_df["gold_sentiment"].eq("")]
print(f"{len(todo)} left to label")

for i, row in enumerate(todo.itertuples(), 1):
    print(f"\n[{i}/{len(todo)}] {row.text}")
    key = input("n / u / p (+?) / s / q > ").strip().lower()
    while key.rstrip("?") not in {*KEYS, "s", "q"}:
        key = input("  use n, u, p, s or q > ").strip().lower()
    if key == "q":
        break
    if key == "s":
        continue
    save_label(row.uid, "gold_sentiment", KEYS[key.rstrip("?")])
    if key.endswith("?"):
        save_label(row.uid, "notes", "sentiment unsure", append=True)

In [ ]:
# Progress and label balance by source
eval_df = pd.read_csv(EVAL_PATH, dtype=str, keep_default_na=False)
labels = eval_df["gold_sentiment"].replace("", "(unlabeled)")
display(pd.crosstab(eval_df["source"], labels, margins=True))
print("Flagged unsure:", eval_df["notes"].str.contains("sentiment unsure").sum())